In [0]:
-- Goalie - Save Percentage by Shot Type
-- Which shot types beat each goalie.
-- Save % is from play-by-play shots on goal, so it can differ slightly from the official figure.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    'MTL' AS team,                   -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,              -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type,                  -- 2 = regular season, 3 = playoffs
    CAST(NULL AS STRING) AS player   -- e.g. 'Nick Suzuki'; NULL = every player
),
player_games AS (                        -- one row per goalie per game dressed (game rosters)
  SELECT r.game_id, r.player_id, p.full_name, r.position_code, r.sweater_number,
         r.team_id, t.abbrev AS team, o.abbrev AS opponent, r.team_id = g.home_team_id AS is_home,
         g.season, g.game_date, g.home_team_id,
         CASE WHEN IF(r.team_id = g.home_team_id, g.home_final_score > g.away_final_score,
                                                  g.away_final_score > g.home_final_score) THEN 'W'
              WHEN g.game_type = 2 AND g.last_period_type IN ('OT', 'SO') THEN 'OTL'
              ELSE 'L' END AS result
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game   g ON r.game_id   = g.game_id
  JOIN nhl.gold.dim_player p ON r.player_id = p.player_id
  JOIN nhl.gold.dim_team   t ON r.team_id   = t.team_id
  JOIN nhl.gold.dim_team   o ON o.team_id   = IF(r.team_id = g.home_team_id, g.away_team_id, g.home_team_id)
  CROSS JOIN params pr
  WHERE g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
    AND (pr.player IS NULL OR lower(p.full_name) = lower(pr.player))
    AND r.position_code = 'G'
),
goalie_shots AS (                        -- every shot on goal those goalies faced
  SELECT pg.full_name, pg.player_id, pg.game_id, pg.game_date, pg.opponent, pg.result,
         f.type_desc_key = 'goal' AS is_goal, f.shot_type, f.zone_code, f.period_number, f.period_type,
         SQRT(POW(89 - ABS(f.x_coord), 2) + POW(f.y_coord, 2)) AS distance_ft,
         IF(f.event_owner_team_id = pg.home_team_id, f.situation_code, reverse(f.situation_code)) AS owner_situation,
         CASE WHEN substr(owner_situation, 1, 1) = '0' THEN 'EN'      -- opponent net empty
              WHEN substr(owner_situation, 3, 1) > substr(owner_situation, 2, 1)
               AND substr(owner_situation, 4, 1) = '1' THEN 'PP'      -- more skaters, own goalie in
              WHEN substr(owner_situation, 3, 1) < substr(owner_situation, 2, 1) THEN 'SH'
              ELSE 'EV' END AS strength
  FROM nhl.gold.fact_event f
  JOIN player_games pg ON f.game_id = pg.game_id AND f.goalie_in_net_id = pg.player_id
  WHERE f.type_desc_key IN ('goal', 'shot-on-goal')
    AND COALESCE(f.period_type, '') <> 'SO'    -- shootout attempts aren't stats
)
SELECT full_name,
       shot_type,
       COUNT(*)                                             AS shots_faced,
       COUNT_IF(is_goal)                                    AS goals_against,
       COUNT_IF(NOT is_goal)                                AS saves,
       ROUND(1 - COUNT_IF(is_goal) / COUNT(*), 3)           AS save_pct
FROM goalie_shots
WHERE shot_type IS NOT NULL
GROUP BY full_name, shot_type
ORDER BY full_name, shots_faced DESC;